# Exploring Sales and Product Data in Northwind2025
This notebook walks through a series of T-SQL queries executed against the Northwind2025 database. Each query demonstrates key concepts such as joins, aggregations, scalar and multi-record subqueries, and filtering techniques.

## 1. Basic Join with Calculated Column
**Purpose**: Retrieve order details with a calculated line total.

**Explanation**: This query joins `OrderDetails` with `Orders` and calculates the total price per line item, factoring in quantity and discount.

In [ ]:
SELECT ord.CustomerID
      ,ord.OrderID
      ,od.ProductID
      ,od.UnitPrice * od.Quantity * (1 + od.Discount) as 'LineTotal'
FROM sales.OrderDetails od
    INNER JOIN sales.Orders ord
     ON od.OrderID = ord.OrderID;

## 2. Count of Joined Rows
**Purpose**: Count how many rows result from the join.

**Explanation**: This query uses `COUNT(*)` to determine the total number of records from the join between `OrderDetails` and `Orders`.

In [ ]:
SELECT count(*) as 'Records'
FROM sales.OrderDetails od
    INNER JOIN sales.Orders ord
     ON od.OrderID = ord.OrderID;

## 3. Filter by Average Line Total
**Purpose**: Show only rows where the line total is greater than or equal to the average.

**Explanation**: A scalar subquery calculates the average line total, and the main query filters rows based on this value.

In [ ]:
DECLARE @avgSale float 
SET @avgSale =  (SELECT AVG(UnitPrice * Quantity * (1 + Discount)) FROM sales.OrderDetails)

SELECT ord.CustomerID
      ,ord.OrderID
      ,od.ProductID
      ,od.UnitPrice * od.Quantity * (1 + od.Discount) as 'LineTotal'
FROM sales.OrderDetails od
    INNER JOIN sales.Orders ord
     ON od.OrderID = ord.OrderID
WHERE od.UnitPrice * od.Quantity * (1 + od.Discount) >= @avgSale;

## 4. Count Above and Below Average
**Purpose**: Compare how many records are above or below the average line total.

**Explanation**: Two separate queries count records based on whether they are above or below the average line total.

In [ ]:
DECLARE @avgSale float 
SET @avgSale =  (SELECT AVG(UnitPrice * Quantity * (1 + Discount)) FROM sales.OrderDetails)

-- Above Average
SELECT count(*) as 'Records'
FROM sales.OrderDetails od
    INNER JOIN sales.Orders ord
     ON od.OrderID = ord.OrderID
WHERE od.UnitPrice * od.Quantity * (1 + od.Discount) >= @avgSale;

-- Below Average
SELECT count(*) as 'Records'
FROM sales.OrderDetails od
    INNER JOIN sales.Orders ord
     ON od.OrderID = ord.OrderID
WHERE od.UnitPrice * od.Quantity * (1 + od.Discount) < @avgSale;

## 5. Difference from Average
**Purpose**: Show how much each line total differs from the average.

**Explanation**: Adds a column showing the difference between each line total and the average sale value.

In [ ]:
DECLARE @avgSale float 
SET @avgSale =  (SELECT AVG(UnitPrice * Quantity * (1 + Discount)) FROM sales.OrderDetails)

SELECT ord.CustomerID
      ,ord.OrderID
      ,od.ProductID
      ,od.UnitPrice * od.Quantity * (1 + od.Discount) as 'LineTotal'
      ,(od.UnitPrice * od.Quantity * (1 + od.Discount)) - @avgSale as 'Difference'
FROM sales.OrderDetails od
    INNER JOIN sales.Orders ord
     ON od.OrderID = ord.OrderID;

## 6. Total Difference from Average
**Purpose**: Aggregate the total difference from average across all records.

**Explanation**: Wraps the previous query in a subquery and sums the `Difference` column.

In [ ]:
DECLARE @avgSale float 
SET @avgSale =  (SELECT AVG(UnitPrice * Quantity * (1 + Discount)) FROM sales.OrderDetails)

SELECT sum(orders.Difference) as 'TotalDiff'
FROM (
    SELECT ord.CustomerID
          ,ord.OrderID
          ,od.ProductID
          ,od.UnitPrice * od.Quantity * (1 + od.Discount) as 'LineTotal'
          ,(od.UnitPrice * od.Quantity * (1 + od.Discount)) - @avgSale as 'Difference'
    FROM sales.OrderDetails od
        INNER JOIN sales.Orders ord
         ON od.OrderID = ord.OrderID
) as orders;

## 7. Multi-record Subquery: Products from Canadian Suppliers
**Purpose**: Find products supplied by vendors in Canada.

**Explanation**: Uses a subquery to filter products based on supplier country.

In [ ]:
SELECT *
FROM prod.Products
WHERE SupplierID in (SELECT SupplierID FROM prod.Suppliers WHERE Country in ('Canada'));

## 8. Subquery in Join
**Purpose**: Join products with Canadian suppliers using a subquery.

**Explanation**: The subquery is used as a derived table in the join.

In [ ]:
SELECT p.*
FROM prod.Products p
    INNER JOIN (SELECT SupplierID FROM prod.Suppliers WHERE Country in ('Canada')) s
     ON p.SupplierID = s.SupplierID;

## 9. Correlated Subquery: Products Not Sold in 2023
**Purpose**: Find products that were not sold in the year 2023.

**Explanation**: Uses a correlated subquery to exclude products that appear in 2023 sales.

In [ ]:
SELECT p.ProductID 
      ,p.ProductName
FROM prod.Products p
WHERE NOT EXISTS (
    SELECT od.ProductID
    FROM sales.OrderDetails od
        INNER JOIN sales.Orders ord
         ON od.OrderID = ord.OrderID
    WHERE year(ord.OrderDate) = 2023
     AND p.ProductID = od.ProductID
);

## 10. Alternative: NOT IN with Distinct
**Purpose**: Another way to find products not sold in 2023.

**Explanation**: Uses `NOT IN` with a distinct list of product IDs from 2023 sales.

In [ ]:
SELECT p.ProductID 
      ,p.ProductName
FROM prod.Products p
WHERE p.ProductID not in (
    SELECT distinct od.ProductID
    FROM sales.OrderDetails od
        INNER JOIN sales.Orders ord
         ON od.OrderID = ord.OrderID
    WHERE year(ord.OrderDate) = 2023
);